## Loading the temperature forecast: test with Zurich

In stage 2, part B, temperature enters the regression as a feature. To avoid leakage, we do not use the
measured temperature but the **forecast issued 24 hours earlier** (`temperature_2m_previous_day1` from the
Open-Meteo Previous Runs API, model `gfs_seamless`). This information would actually have been available
at forecast time.

Before loading all cities and the full period, we test the request with Zurich for two days in April 2021.
We check which columns are returned, whether the timestamps are in UTC and whether the values look plausible.

In [8]:
import json
import urllib.parse
import urllib.request

import pandas as pd

# Base URL of the Previous Runs API
BASE_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"


def fetch_temperature(lat, lon, start, end, verbose=True):
    # Loads the analysed temperature and the forecast issued 24 h earlier,
    # timestamps in UTC
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,temperature_2m_previous_day1",
        "models": "gfs_seamless",
        "start_date": start,
        "end_date": end,
        "timezone": "GMT",
    }
    url = BASE_URL + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        data = json.load(response)

    # Print the raw structure only if requested
    if verbose:
        print("Top-level keys:", list(data.keys()))
        print("Units:", data["hourly_units"])

    # Convert hourly values to a DataFrame with a UTC index
    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return df.set_index("time")


# Test: Zurich, two days
zurich = fetch_temperature(47.37, 8.54, "2021-04-01", "2021-04-02")
print(zurich.shape)
zurich.head(8)

Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Units: {'time': 'iso8601', 'temperature_2m': '°C', 'temperature_2m_previous_day1': '°C'}
(48, 2)


,temperature_2m,temperature_2m_previous_day1
time,,
2021-04-01 00:00:00+00:00,11.3,10.9
2021-04-01 01:00:00+00:00,10.9,10.5
2021-04-01 02:00:00+00:00,10.3,10.1
2021-04-01 03:00:00+00:00,9.9,10.2
2021-04-01 04:00:00+00:00,9.3,9.9
2021-04-01 05:00:00+00:00,9.0,9.8
2021-04-01 06:00:00+00:00,9.5,10.3
2021-04-01 07:00:00+00:00,11.2,11.9


The request works as expected:

- **Format:** one row per hour, two columns: `temperature_2m` (analysed temperature) and
  `temperature_2m_previous_day1` (forecast issued 24 h earlier). Two days give 48 rows.
- **Time:** the index is in UTC and starts at 00:00, matching the UTC grid of the load data.
- **Plausibility:** values between roughly 9 and 12 °C fit a mild early-April night in Zurich.
  The forecast stays close to the analysis but deviates by up to about 0.8 °C, as expected for a real forecast.

One point to settle before merging: the temperature is an instantaneous value at the timestamp, while each
load value is an hourly average labelled at the end of its interval. We will decide on the alignment when
building the full feature table.

Next step: load the full period (April 2021 to end of 2025) for several Swiss cities and build a
population-weighted average.

## Full period for Zurich

The forecast variable is available from April 2021, so we load Zurich from 1 April 2021 to 31 December 2025.
To keep each request small and robust, we fetch one calendar year at a time and concatenate the pieces.

Afterwards we check that the result is a complete hourly UTC grid: no duplicate timestamps, no gaps
and no missing values in the forecast column.

In [9]:
# Yearly request windows from April 2021 to the end of 2025
windows = [("2021-04-01", "2021-12-31")] + [
    (f"{year}-01-01", f"{year}-12-31") for year in range(2022, 2026)
]

# Fetch each window and concatenate
zurich = pd.concat(
    [fetch_temperature(47.37, 8.54, start, end, verbose=False) for start, end in windows]
)

# Expected complete hourly grid in UTC
expected = pd.date_range("2021-04-01 00:00", "2025-12-31 23:00", freq="h", tz="UTC")

print("Rows:", f"{len(zurich):,}".replace(",", "'"))
print("Expected rows:", f"{len(expected):,}".replace(",", "'"))
print("Duplicate timestamps:", zurich.index.duplicated().sum())
print("Missing timestamps:", len(expected.difference(zurich.index)))

# Check for missing values per column
print("Missing values per column:")
print(zurich.isna().sum())
zurich.describe().round(1)

Rows: 41'664
Expected rows: 41'664
Duplicate timestamps: 0
Missing timestamps: 0
Missing values per column:
temperature_2m                    0
temperature_2m_previous_day1    492
dtype: int64


,temperature_2m,temperature_2m_previous_day1
count,41664.0,41172.0
mean,11.4,11.4
std,7.8,7.7
min,-14.6,-12.3
25%,5.7,5.7
50%,11.0,11.0
75%,16.6,16.6
max,36.4,37.4


The hourly UTC grid is complete: 41'664 rows from April 2021 to the end of 2025, no duplicate and no missing
timestamps. The analysed temperature has no gaps. Both columns have almost identical distributions
(mean 11.4 °C, median 11.0 °C), and the range from about -15 °C to 37 °C is plausible for Zurich.

The **forecast column**, however, has 492 missing values (about 1.2 %). Before choosing how to handle them,
we check whether they are isolated hours or longer blocks.

In [10]:
# Mark missing forecast values and group consecutive missing hours into blocks
missing = zurich["temperature_2m_previous_day1"].isna()
block_id = (missing != missing.shift()).cumsum()

blocks = (
    zurich[missing]
    .assign(block=block_id[missing])
    .reset_index()
    .groupby("block")["time"]
    .agg(start="min", end="max", hours="count")
    .reset_index(drop=True)
)

print("Number of missing blocks:", len(blocks))
blocks.sort_values("hours", ascending=False).head(15)

Number of missing blocks: 1


,start,end,hours
0,2023-12-30 00:00:00+00:00,2024-01-19 11:00:00+00:00,492


All 492 missing forecast values form a single block of about 20 days, from 30 December 2023 to 19 January 2024.
This rules out interpolation, and filling the gap with the analysed temperature would leak information that
is not available at forecast time.

The gap lies exactly at the boundary between training (2023) and validation (2024). A clean fallback is the
forecast issued two days earlier (`temperature_2m_previous_day2`): it was also available at forecast time,
just slightly less accurate. We check whether it covers the gap.

In [11]:
# Request the forecast issued 48 h earlier for the gap window
params = {
    "latitude": 47.37,
    "longitude": 8.54,
    "hourly": "temperature_2m_previous_day2",
    "models": "gfs_seamless",
    "start_date": "2023-12-29",
    "end_date": "2024-01-20",
    "timezone": "GMT",
}
url = BASE_URL + "?" + urllib.parse.urlencode(params)
with urllib.request.urlopen(url) as response:
    data = json.load(response)

day2 = pd.DataFrame(data["hourly"])
day2["time"] = pd.to_datetime(day2["time"], utc=True)
day2 = day2.set_index("time")["temperature_2m_previous_day2"]

# Check coverage exactly on the missing hours
gap_index = zurich.index[missing]
print("Missing hours:", len(gap_index))
print("Covered by previous_day2:", day2.reindex(gap_index).notna().sum())

Missing hours: 492
Covered by previous_day2: 0


The 48-hour forecast is missing for exactly the same hours. The gap is therefore not a single missing run but
a gap in the archive of the GFS model for these 20 days. As a fallback we check whether the archives of other
weather models (ICON, ECMWF) contain day-ahead forecasts for this period; such forecasts were also available
at forecast time, so using them causes no leakage.

In [12]:
# Check day-ahead forecast coverage of other models on the missing hours
for model in ["icon_seamless", "ecmwf_ifs025", "best_match"]:
    params = {
        "latitude": 47.37,
        "longitude": 8.54,
        "hourly": "temperature_2m_previous_day1",
        "models": model,
        "start_date": "2023-12-29",
        "end_date": "2024-01-20",
        "timezone": "GMT",
    }
    url = BASE_URL + "?" + urllib.parse.urlencode(params)
    try:
        with urllib.request.urlopen(url) as response:
            data = json.load(response)
        series = pd.DataFrame(data["hourly"])
        series["time"] = pd.to_datetime(series["time"], utc=True)
        series = series.set_index("time")["temperature_2m_previous_day1"]
        print(f"{model}: covered {series.reindex(gap_index).notna().sum()} of {len(gap_index)} hours")
    except Exception as error:
        # Some models may not be available in the archive for this period
        print(f"{model}: request failed ({error})")

icon_seamless: covered 0 of 492 hours
ecmwf_ifs025: covered 0 of 492 hours
best_match: covered 0 of 492 hours


None of the archived models (GFS, ICON, ECMWF, best match) has day-ahead forecasts for this period, so the gap
is an outage of the forecast archive itself rather than of a single model.

**Decision.** We exclude the 492 affected hours instead of filling them. Interpolation is impossible over 20 days,
the analysed temperature would leak, and a climatological average would be an artificial, much weaker feature.
The cost is small: 48 hours of training data (30–31 December 2023) and 444 hours of validation data
(1–19 January 2024, about 5 % of the validation year).

For a fair comparison, the regression **without** temperature is trained and evaluated on exactly the same
hours as the regression **with** temperature.

## Population-weighted Swiss temperature

Electricity demand depends on the temperature where people live and work, so a single city is not enough.
We use the eight largest Swiss agglomerations, which together cover the main climatic regions: the Swiss
Plateau, the Lake Geneva region, north-western Switzerland, central and eastern Switzerland, and Ticino south
of the Alps. Each city is weighted by the population of its agglomeration (Swiss Federal Statistical Office,
2025, via [top10cities.net](https://top10cities.net/country/switzerland-agglo.php)). Winterthur and Zug are
left out because their climate is practically the same as Zurich's.

For each city we load the day-ahead forecast from April 2021 to the end of 2025 and check for missing values.

In [13]:
# Agglomerations: latitude, longitude, population 2025 (BFS)
cities = {
    "Zurich":    (47.37, 8.54, 1_498_745),
    "Geneva":    (46.20, 6.15,   632_065),
    "Basel":     (47.56, 7.59,   580_603),
    "Lausanne":  (46.52, 6.63,   462_325),
    "Bern":      (46.95, 7.45,   443_509),
    "Lucerne":   (47.05, 8.31,   253_460),
    "St. Gallen": (47.42, 9.37,  183_845),
    "Lugano":    (46.00, 8.95,   154_802),
}

# Load the day-ahead forecast for each city, one column per city
forecasts = {}
for name, (lat, lon, _) in cities.items():
    pieces = [fetch_temperature(lat, lon, start, end, verbose=False) for start, end in windows]
    forecasts[name] = pd.concat(pieces)["temperature_2m_previous_day1"]
city_temp = pd.DataFrame(forecasts)

# Population weights normalised to sum to one
weights = pd.Series({name: pop for name, (_, _, pop) in cities.items()})
weights = weights / weights.sum()

print("Weights:")
print(weights.round(3))
print("\nMissing values per city:")
print(city_temp.isna().sum())

Weights:
Zurich        0.356
Geneva        0.150
Basel         0.138
Lausanne      0.110
Bern          0.105
Lucerne       0.060
St. Gallen    0.044
Lugano        0.037
dtype: float64

Missing values per city:
Zurich        492
Geneva        492
Basel         492
Lausanne      492
Bern          492
Lucerne       492
St. Gallen    492
Lugano        492
dtype: int64


The weights are each agglomeration's share of the combined population of the eight agglomerations
(about 4.2 million), so they sum to one. Zurich dominates with 36 %, followed by Geneva, Basel, Lausanne and Bern.

Every city has exactly 492 missing values, which confirms that the gap is an outage of the forecast archive
affecting all locations at once rather than a local problem.

In [14]:
# Check that all cities miss exactly the same hours
same_gap = city_temp.isna().eq(city_temp["Zurich"].isna(), axis=0).all().all()
print("Same missing hours in all cities:", same_gap)

# Population-weighted Swiss temperature (missing where the archive has no forecast)
swiss_temp = city_temp.mul(weights, axis=1).sum(axis=1, min_count=len(weights))
swiss_temp.name = "temp_forecast"

# Correlation between cities and with the weighted average
print("\nCorrelation with the weighted average:")
print(city_temp.corrwith(swiss_temp).round(3))
print("\nLowest pairwise correlation between cities:", city_temp.corr().min().min().round(3))
swiss_temp.describe().round(1)

Same missing hours in all cities: True

Correlation with the weighted average:
Zurich        0.995
Geneva        0.986
Basel         0.992
Lausanne      0.988
Bern          0.993
Lucerne       0.990
St. Gallen    0.982
Lugano        0.949
dtype: float64

Lowest pairwise correlation between cities: 0.927


count    41172.0
mean        11.6
std          7.5
min        -10.1
25%          6.0
50%         11.2
75%         16.7
max         36.2
Name: temp_forecast, dtype: float64

For each hour $t$, the Swiss temperature is the population-weighted average of the eight city forecasts,

$$
T_{\mathrm{CH}}(t) = \sum_{c} w_c \, T_c(t), \qquad w_c = \frac{P_c}{\sum_{c'} P_{c'}},
$$

where $T_c(t)$ is the day-ahead forecast at the grid point of city $c$ and $P_c$ is the population of its
agglomeration. Each city is represented by a single grid point, not by an average over its whole agglomeration;
the agglomeration enters only through its population weight.

All cities miss exactly the same hours, so the weighted average is simply undefined during the archive gap.

The Pearson correlations show that the cities move almost in lockstep: each city correlates with the weighted
average at 0.95 or more, and even the least similar pair of cities still reaches 0.93. Lugano, south of the
Alps, deviates most. Each city is part of the average itself, which inflates its correlation with it, so the
pairwise value is the more honest measure. Part of the similarity also comes from the shared seasonal and daily
cycle; deviations from typical weather differ more between regions. Still, eight separate city features would
be highly collinear, which supports using a **single population-weighted temperature**.

## Saving the processed temperature

We store the population-weighted day-ahead temperature as a Parquet file next to the processed load data,
so that later notebooks can use it without querying the API again. The archive gap is kept as missing values;
the hours are excluded when the feature table is built.

In [15]:
from pathlib import Path

# Save the Swiss temperature forecast (hourly, UTC, gap as missing values)
out_path = Path("../data/processed/temp_forecast_2021_2025.parquet")
swiss_temp.to_frame().to_parquet(out_path)

# Read back to check that the file is complete
check = pd.read_parquet(out_path)
print("Saved to:", out_path)
print("Rows:", f"{len(check):,}".replace(",", "'"))
print("Missing values:", check["temp_forecast"].isna().sum())
print("Index time zone:", check.index.tz)

Saved to: ..\data\processed\temp_forecast_2021_2025.parquet
Rows: 41'664
Missing values: 492
Index time zone: UTC


The file contains the complete hourly UTC grid from April 2021 to the end of 2025 (41'664 rows), with the
492 hours of the archive gap as missing values. Next step: align the instantaneous temperature with the
hourly average load (labelled at the end of each interval), build the feature table and compare the
regression with and without temperature on identical hours.